# KIST PacBio Full-Length 16S · P92 / 92개 중 PC·NC 제외 (90개)
원본 Excel 기준 **상단 92개 (P92)**와 **하단 18개 (P18)**는 별도 프로젝트이며, 함께 분석하지 않습니다.

**이 노트북은 P92 / no_controls / 예상 샘플 90개만 분석**합니다.

- P92: KCT / K902 / K905의 W1–W3, PC 및 NC 각 1개 (총 92개)
- P18: KUW / KUD / KCR / KDR 각 4개, PC 및 NC 각 1개 (총 18개)
- P92 제외 대상 3개: K905_W1_10 (FKDN260619042-1A), KCT_W3_10 (FKDN260619082-1A), K902_W3_02 (FKDN260619084-1A).
- PC, NC는 프로젝트마다 별개의 FKDN ID입니다.

**분석**: **PC와 NC는 절대 분석하지 않습니다.** FASTQ 분석 시 manifest 단계부터 제외한 후 각 노트북별 DADA2 CCS → ASV를 생성합니다. 기존 결과 ZIP의 ASV table을 재사용할 경우에는 NC/PC를 ASV 테이블에서 제거하고 다양성 지표를 다시 계산합니다.
세 프로젝트 제외 샘플 3개는 **P92 87개 분석에만** 반영합니다. 결과는 프로젝트별 Google Drive 폴더에 개별 저장합니다.

**입력 주의:** 기존 2.17GB ZIP이 원시 FASTQ를 포함하는지는 미확인입니다. FASTQ와 검증된 primer가 없으면 신규 DADA2 분석은 실행할 수 없습니다. 양쪽 프로젝트 PC/NC 샘플을 구별할 수 없는 ASV table은 분석을 중지합니다.

**2026-11-12 FASTQ-only update:** Execute from FASTQ inside ZIP or Drive/raw_fastq; no ASV table assumed. The analyzed input never includes PC/NC. Provide the verified amplification forward primer before QIIME 2 installation.

**Primer 확정:** 27F `AGRGTTYGATYMTGGCTCAG` (5'->3'); 1492R `GYTACCTTGTTACGACTT` (5'->3'). DADA2의 `--p-adapter`에는 reverse-complement가 아닌 **표에 표기된 1492R 방향 그대로** 전달합니다. 문헌의 RGYTAC... 1492R 변형과 1-base 차이가 있으므로 primer-trimming retention을 확인합니다.

**Raw data routing:** F003 → P92. Drive에 압축 해제한 01.RawData의 `*_withprimer.fastq.gz`만 사용합니다. 같은 샘플의 00.CleanData를 중복 투입하지 않습니다. QIIME2 YAML은 공식 URL에서 원본을 내려받아 로컬 경로로 설치합니다.


### DB-free phase 1 (F003, 90 samples)
SILVA classification is deferred. Runs primer-aware DADA2 CCS on **90 sample-specific F003 01.RawData FASTQs only**, exports ASV table, representative sequences and denoising stats, then independently computes diversity. All PC/NC excluded. The complete raw source ZIP remains untouched. ASV QZA artifacts are saved for later SILVA classification without re-denoising.


In [ ]:
# 0. Project- and scenario-specific configuration
DRIVE_PROJECT_ID = "1070GDz6WnKV23Ygzgs08idwO1p4-xYm5"
DRIVE_METADATA_XLSX_ID = "1_USCzdwWfQ614tjzDocbGDcM5aZp-YBD"
DRIVE_EXISTING_ZIP_ID = "1rPpGDmZHDcjNAm0UmjP1kZl5VdGB9478"
DRIVE_FASTQ_FOLDER_ID = "1tJHfddKf3jlYD-xGJSc7MB_uq9bW8Yyi"  # F003 extracted_original; must contain 01.RawData
DRIVE_REPORT_FOLDER_ID = "1lRWH_xxawuIVtxliDQ0xtnT5n8XL19D0"
DRIVE_REFERENCE_FOLDER_ID = "1Ed4eWpq7kCTaJG0A81PZ42tfX2mtga7-"
GITHUB_REF = "feat/full-length-16s-hifi-pipeline-20261112"
PROJECT = "P92"
RUN_SCENARIO = "no_controls"
EXPECTED_SAMPLES = 90
DOWNLOAD_VENDOR_ZIP = False  # Only use verified extracted Drive files, never redownload ZIP.
DOWNLOAD_RAW_FASTQ = True  # Fetch from the exact project 01.RawData, not 00.CleanData.
RUN_NEW_DADA2 = True  # Perform actual FASTQ-to-ASV inference (requires verified primer).
INSTALL_QIIME = True  # Install 2026.10; resource-intensive in Colab.
QIIME_CONDA_ENV = "rachis-qiime2-2026.10"
SKIP_TAXONOMY = True  # Phase 1: SILVA not used, export ASVs and diversity first
FORWARD_PRIMER_5TO3 = "AGRGTTYGATYMTGGCTCAG"  # 27F; IUPAC degeneracy retained; exact supplied laboratory sheet.
REVERSE_PRIMER_5TO3 = "GYTACCTTGTTACGACTT"   # 1492R, supplied 5'-3' (do NOT reverse-complement for denoise-ccs adapter).
# IMPORTANT: a published 1492R variant starts RGYTAC..., while this laboratory sheet starts GYTAC... .
# Primer match/retention QC must be checked before scientific PASS.
from pathlib import Path
WORK = Path("/content/kist_full_length_16s/P92/no_controls")
WORK.mkdir(parents=True, exist_ok=True)
print("Running ONLY:",PROJECT,RUN_SCENARIO, "Expected samples:", EXPECTED_SAMPLES)


In [ ]:
# 1. Authenticate Drive and fetch Python code from GitHub
import os, sys, json, io, shutil, subprocess, requests, zipfile
from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive", force_remount=False)
subprocess.run([sys.executable,"-m","pip","install","-q","pandas","numpy","scipy","matplotlib","openpyxl"],check=True)
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import google.auth, pandas as pd
creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
drive_api = build("drive","v3",credentials=creds,cache_discovery=False)
def list_drive(folder):
    files, token = [], None
    while True:
        page = drive_api.files().list(q="'"+folder+"' in parents and trashed = false",
               fields="nextPageToken,files(id,name,size,mimeType)",pageSize=1000,pageToken=token).execute()
        files.extend(page.get("files",[]))
        token=page.get("nextPageToken")
        if not token: break
    return files
def download_from_drive(file_id, target):
    target = Path(target)
    if target.exists() and target.stat().st_size > 0:
        print("Cached:",target.name); return target
    target.parent.mkdir(parents=True,exist_ok=True)
    req=drive_api.files().get_media(fileId=file_id)
    with target.open("wb") as f:
        dl=MediaIoBaseDownload(f,req,chunksize=16*1024*1024)
        done=False
        while not done:
            _,done=dl.next_chunk()
    return target
def github_file(path):
    url="https://raw.githubusercontent.com/kimtk94/Codex/"+GITHUB_REF+"/full_length_16s/"+path
    response=requests.get(url,timeout=60); response.raise_for_status()
    result=WORK/path; result.parent.mkdir(parents=True,exist_ok=True)
    result.write_bytes(response.content)
    return result
MAIN=github_file("src/full16s.py")
SUBSETS=github_file("src/subset_analyses.py")
PROJECT_SCRIPT=github_file("src/project_splits.py")
FASTQ_SCRIPT=github_file("src/fastq_inputs.py")
QIIME_SCRIPT=github_file("scripts/run_qiime2.sh")
print("GitHub files retrieved",MAIN.name,SUBSETS.name,PROJECT_SCRIPT.name,QIIME_SCRIPT.name)

In [ ]:
# 2. Verify original Excel 110 rows, split first 92 and last 18 by row order.
XLSX = download_from_drive(DRIVE_METADATA_XLSX_ID, WORK/"metadata/Pacbio_Analysis_Form.xlsx")
META = WORK/"metadata/sample_metadata_110.tsv"
subprocess.run([sys.executable,str(MAIN),"samples",str(XLSX),str(META)],check=True)
SPLIT_DIR = WORK/"metadata"/"projects"
subprocess.run([sys.executable,str(PROJECT_SCRIPT),"split","--metadata",str(META),
                "--output-dir",str(SPLIT_DIR)],check=True)
SCOPED_META = SPLIT_DIR/(PROJECT+"_metadata.tsv")
metadata = pd.read_csv(SCOPED_META,sep="\t",dtype=str).fillna("")
assert len(metadata)==(92 if PROJECT=="P92" else 18)
assert metadata["control"].isin(["PC","NC"]).sum()==2
assert metadata["control"].value_counts().get("PC",0)==1
assert metadata["control"].value_counts().get("NC",0)==1
print("Project",PROJECT, "reference rows",len(metadata),"PC/NC",metadata["control"].value_counts().to_dict())
# This metadata is for provenance only. The analysis manifest MUST exclude PC/NC.
analysis_metadata = metadata.loc[~metadata["control"].isin(["PC","NC"])].copy()
if PROJECT == "P92" and RUN_SCENARIO == "no_controls_no_three":
    EXCLUDE_FKDN = {"FKDN260619042-1A","FKDN260619082-1A","FKDN260619084-1A"}
    analysis_metadata = analysis_metadata.loc[~analysis_metadata["novogene-id"].isin(EXCLUDE_FKDN)].copy()
ANALYSIS_META = WORK/"metadata"/"analysis_metadata_no_PC_NC.tsv"
assert len(analysis_metadata) == EXPECTED_SAMPLES, f"Expected {EXPECTED_SAMPLES}, got {len(analysis_metadata)}"
assert not analysis_metadata["sample-name"].str.upper().isin(["PC","NC"]).any()
analysis_metadata.to_csv(ANALYSIS_META, sep="\t", index=False)
print("Analysis samples:",len(analysis_metadata),"controls:",analysis_metadata["control"].value_counts().to_dict())
display(analysis_metadata[["excel_order","sample-id","sample-name","novogene-id","cohort","week","control"]])


In [ ]:
# 3. Project-specific control IDs and three validated P92 exclusions
CONTROLS=metadata.loc[metadata["control"].isin(["PC","NC"]),["sample-id","novogene-id","control"]]
print("Control IDs:",CONTROLS.to_dict("records"))
P92_EXCLUSIONS=[
    ("K905_W1_10","FKDN260619042-1A"),
    ("KCT_W3_10","FKDN260619082-1A"),
    ("K902_W3_02","FKDN260619084-1A")
]
if PROJECT == "P92":
    for name,fkdn in P92_EXCLUSIONS:
        x = metadata.loc[metadata["novogene-id"]==fkdn]
        assert len(x)==1 and x["sample-name"].item()==name, "Excluded sample mismatch: "+fkdn
    print("P92 confirmed three exclusions:",P92_EXCLUSIONS)
else:
    assert not any(metadata["novogene-id"].isin([x[1] for x in P92_EXCLUSIONS]))
    print("P18 never excludes the three P92 samples.")
print("Analysis preset:",RUN_SCENARIO,"target",EXPECTED_SAMPLES)


In [ ]:
# 4. List project-specific extracted Google Drive files, only 01.RawData/*_withprimer.fastq.gz.
ZIP = WORK / "vendor" / "Result_X209SC26096091-Z01-ARCHIVE_16S.zip"
FEATURE = WORK / "qiime2_outputs" / "feature-table.tsv"
TAXONOMY = WORK / "qiime2_outputs" / "taxonomy_export" / "taxonomy.tsv"
def list_original_raw_fastqs(extracted_folder_id):
    stack=[(extracted_folder_id, "")]
    found=[]
    while stack:
        folder_id, path = stack.pop()
        children=list_drive(folder_id)
        for item in children:
            name=item["name"]
            subpath=(path+"/"+name).strip("/")
            if item.get("mimeType")=="application/vnd.google-apps.folder":
                # Never enter processed reads (00.CleanData) or report subtrees.
                if ("00.CleanData" in subpath or "report_" in subpath or
                    "02.Result" in subpath):
                    continue
                if (not path or "01.RawData" in subpath or
                    name.startswith("result_X")):
                    stack.append((item["id"],subpath))
            elif ("/01.RawData/" in "/"+subpath and
                  name.lower().endswith("_withprimer.fastq.gz")):
                found.append({**item, "source_path":subpath})
    names=[x["name"] for x in found]
    if len(names)!=len(set(names)):
        raise ValueError("Duplicate raw FASTQ filenames in extracted Drive folder")
    return sorted(found,key=lambda x:x["name"])
RAW_ITEMS=list_original_raw_fastqs(DRIVE_FASTQ_FOLDER_ID)
EXPECTED_RAW=92
print("Source project:",PROJECT,"archive:", "F003")
print("Raw FASTQs:",len(RAW_ITEMS),"/",EXPECTED_RAW)
print("First samples:",[x["name"] for x in RAW_ITEMS[:5]])
if len(RAW_ITEMS)!=EXPECTED_RAW:
    raise RuntimeError("Drive upload incomplete or incorrect source. Expected "+str(EXPECTED_RAW)+
                       " source FASTQs but found "+str(len(RAW_ITEMS))+
                       ". Check F003/F004 extracted folders and rerun this cell.")
print("PASS: correct project archive, no 00.CleanData files included.")


In [ ]:
# 5. Download project-specific raw HiFi FASTQ only; build 90/87/16 sample manifest.
FASTQ_DIR=WORK/"raw_fastq"
PREP_OUT=WORK/"prepared_inputs"
MANIFEST=PREP_OUT/"manifest.tsv"
sys.path.insert(0,str(FASTQ_SCRIPT.parent))
from fastq_inputs import classify
approved=analysis_metadata.to_dict("records")
selected=[item for item in RAW_ITEMS if classify(item["name"],approved) is not None]
if len(selected)!=EXPECTED_SAMPLES:
    missing=set(analysis_metadata["sample-name"])-set(x["name"].replace("_withprimer.fastq.gz","") for x in selected)
    raise ValueError(f"Eligible FASTQ count {len(selected)} differs from expected {EXPECTED_SAMPLES}. "
                     f"Missing sample names: {sorted(missing)}")
FASTQ_DIR.mkdir(parents=True,exist_ok=True)
for i,item in enumerate(selected,1):
    destination=FASTQ_DIR/item["name"]
    download_from_drive(item["id"],destination)
    if "size" in item and int(item["size"])!=destination.stat().st_size:
        raise IOError("Downloaded FASTQ size mismatch: "+item["name"])
    if i%10==0 or i==len(selected):
        print("FASTQ download:",i,"/",len(selected),flush=True)
cmd=[sys.executable,str(FASTQ_SCRIPT),"--metadata",str(META),
     "--project",PROJECT,"--scenario",RUN_SCENARIO,
     "--output",str(PREP_OUT),"--directory",str(FASTQ_DIR)]
subprocess.run(cmd,check=True)
counts=pd.read_csv(MANIFEST,sep="\t")
assert len(counts)==EXPECTED_SAMPLES,(len(counts),EXPECTED_SAMPLES)
assert set(counts["sample-id"])==set(analysis_metadata["sample-id"])
assert not counts["sample-id"].str.contains("PC__|NC__",regex=True).any()
subprocess.run([sys.executable,str(MAIN),"qc",str(MANIFEST),
                str(WORK/"metadata/fastq_qc.tsv")],check=True)
print("PASS: strict raw FASTQ manifest",PROJECT,RUN_SCENARIO,len(counts))

In [ ]:
# 6. (Optional, heavy) install QIIME 2 2026.10 + denoise PacBio HiFi.
# Direct official environment: https://library.qiime2.org/quickstart/qiime2
# Colab may lack resources for the full environment. GitHub Actions only handles CI.
if RUN_NEW_DADA2 and not FORWARD_PRIMER_5TO3:
    raise ValueError("Set the verified 5-prime amplification primer in FORWARD_PRIMER_5TO3.")
if RUN_NEW_DADA2:
    assert FORWARD_PRIMER_5TO3 == "AGRGTTYGATYMTGGCTCAG", "Unexpected 27F primer"
    assert REVERSE_PRIMER_5TO3 == "GYTACCTTGTTACGACTT", "Unexpected lab-specific 1492R primer"
    print("Primer pair:", FORWARD_PRIMER_5TO3, "/", REVERSE_PRIMER_5TO3)
    print("DADA2 will use both primers as 5'-3' IUPAC strings without reverse-complementing 1492R.")
if RUN_NEW_DADA2 and not MANIFEST.exists():
    raise FileNotFoundError("FASTQ manifest absent. Run cell 5 first.")
if INSTALL_QIIME or RUN_NEW_DADA2:
    import tarfile
    micromamba=WORK/"bin/micromamba"
    if not micromamba.exists():
        url="https://micro.mamba.pm/api/micromamba/linux-64/latest"
        res=requests.get(url,timeout=180);res.raise_for_status()
        with tarfile.open(fileobj=io.BytesIO(res.content),mode="r:bz2") as tar:
            member=next(item for item in tar.getmembers() if item.name=="bin/micromamba")
            micromamba.parent.mkdir(parents=True,exist_ok=True)
            with tar.extractfile(member) as src,micromamba.open("wb") as dst:shutil.copyfileobj(src,dst)
        micromamba.chmod(0o755)
    env={**os.environ,"MAMBA_ROOT_PREFIX":str(WORK/"micromamba")}
    yaml_url="https://raw.githubusercontent.com/qiime2/distributions/refs/heads/dev/2026.10/qiime2/released/rachis-qiime2-linux-64-conda.yml"
    yaml_file=WORK/"bin/rachis-qiime2-linux-64-conda.yml"
    if INSTALL_QIIME:
        response=requests.get(yaml_url, timeout=90)
        response.raise_for_status()
        if "channels:" not in response.text or "dependencies:" not in response.text:
            raise ValueError("Downloaded QIIME 2 environment YAML invalid: channels/dependencies missing")
        yaml_file.parent.mkdir(parents=True,exist_ok=True)
        yaml_file.write_text(response.text,encoding="utf-8")
        print("QIIME2 environment YAML downloaded:",yaml_file,len(response.content),"bytes")
        installed=subprocess.run([str(micromamba),"env","list"],env=env,
                                 capture_output=True,text=True,check=True)
        if QIIME_CONDA_ENV not in installed.stdout:
            subprocess.run([str(micromamba),"create","-y","-n",QIIME_CONDA_ENV,
                            "-f",str(yaml_file)],check=True,env=env)
        else:
            print("Existing QIIME 2 environment found; avoiding reinstall:", QIIME_CONDA_ENV)
    if RUN_NEW_DADA2:
        if not MANIFEST.exists(): raise FileNotFoundError("Upload FASTQs and run cell 5.")
        if not FORWARD_PRIMER_5TO3: raise ValueError("Required actual forward primer not configured.")
        CLASSIFIER=""  # DADA2 ASV inference does not use a taxonomy database.
        if not SKIP_TAXONOMY:
            candidate_qza=[x for x in list_drive(DRIVE_REFERENCE_FOLDER_ID)
                           if x["name"].lower().endswith(".qza")]
            if not candidate_qza:
                raise FileNotFoundError("Taxonomy requested, but no classifier .qza in Drive/references")
            item=candidate_qza[0]
            CLASSIFIER=str(download_from_drive(item["id"],WORK/"references"/item["name"]))
        else:
            print("F003 phase 1: SILVA/Taxonomy skipped. Running DADA2 and diversity only.")
        cmd=[str(micromamba),"run","-n",QIIME_CONDA_ENV,"bash",str(QIIME_SCRIPT),
            str(MANIFEST),str(FEATURE.parent),FORWARD_PRIMER_5TO3,CLASSIFIER,REVERSE_PRIMER_5TO3]
        subprocess.run(cmd,env=env,check=True)
        assert (FEATURE.parent/"table.qza").exists()
        assert (FEATURE.parent/"rep-seqs.qza").exists()
        assert (FEATURE.parent/"stats.qza").exists()
        print("ASV inference artifacts created; classifier not required.")
else:
    print("QIIME 2 optional denoising not requested.")

In [ ]:
# 7. Recompute this project's ASV/alpha/beta/report for ONLY the selected preset.
OUTPUT=WORK/"analysis"
if FEATURE.exists():
    cmd=[sys.executable,str(PROJECT_SCRIPT),"run",
         "--feature-table",str(FEATURE),
         "--metadata",str(META),
         "--project",PROJECT,
         "--scenario",RUN_SCENARIO,
         "--output-dir",str(OUTPUT)]
    if (not SKIP_TAXONOMY) and TAXONOMY.exists():
        cmd.extend(["--taxonomy",str(TAXONOMY)])
    subprocess.run(cmd,check=True)
    summary=json.loads((OUTPUT/"project_run.json").read_text(encoding="utf-8"))
    assert summary["status"]=="PASS" and summary["actual_samples"]==EXPECTED_SAMPLES
    print("PASS:",PROJECT,RUN_SCENARIO, summary)
    print("HTML report:", OUTPUT/"reports"/RUN_SCENARIO/"report.html")
    print("Taxonomy status:", "SKIPPED_BY_DESIGN" if SKIP_TAXONOMY else "REQUESTED")
else:
    raise FileNotFoundError("ASV feature-table.tsv not available. Check vendor ZIP content or upload raw HiFi FASTQ and run DADA2 first.")


In [ ]:
# 8. Upload F003 90-sample DB-free DADA2 ASVs, denoising QC and diversity to separate Drive folder.
from datetime import datetime, timezone
from googleapiclient.http import MediaFileUpload
OUTPUT = WORK/"analysis"
assert (OUTPUT/"project_run.json").exists(), "Run DADA2 and reference-free diversity first"
assert (FEATURE.parent/"table.qza").exists(), "Missing DADA2 ASV table"
assert (FEATURE.parent/"rep-seqs.qza").exists(), "Missing representative ASV sequences"
assert (FEATURE.parent/"stats.qza").exists(), "Missing denoising statistics"
allowed = {".html",".png",".tsv",".json",".csv",".qza",".fasta",".biom"}
roots=[WORK/"metadata",WORK/"prepared_inputs",WORK/"analysis",WORK/"qiime2_outputs"]
files=[f for root in roots if root.exists()
       for f in root.rglob("*")
       if f.is_file() and f.suffix.lower() in allowed
       and f.name!="demux.qza" and "merged_lanes" not in str(f)]
mandatory = {"table.qza","rep-seqs.qza","stats.qza","dna-sequences.fasta","feature-table.tsv","project_run.json"}
missing = mandatory - {x.name for x in files}
if missing:
    raise FileNotFoundError(f"Mandatory DB-free ASV artifacts missing: {sorted(missing)}")
stamp=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
folder=drive_api.files().create(
    body={"name":"F003_P92_90_DADA2_NoDB_"+stamp,
          "mimeType":"application/vnd.google-apps.folder",
          "parents":[DRIVE_REPORT_FOLDER_ID]},
    fields="id").execute()
dest=folder["id"]
saved=[]
for f in files:
    # Preserve QIIME artifacts for later taxonomy. Do not upload raw FASTQs or demux.qza.
    if f.stat().st_size>150*1024*1024:
        print("LARGE FILE SKIPPED (review for manual upload):",f.name,f.stat().st_size)
        continue
    rel=str(f.relative_to(WORK)).replace("/","__")
    mimetype= "application/octet-stream" if f.suffix in {".qza",".biom"} else (
        "image/png" if f.suffix==".png" else (
        "text/html" if f.suffix==".html" else "text/plain"))
    media=MediaFileUpload(str(f),mimetype=mimetype,resumable=True)
    obj=drive_api.files().create(
        body={"name":rel,"parents":[dest]},media_body=media,fields="id,name").execute()
    saved.append({"file":rel,"drive_id":obj["id"],"bytes":f.stat().st_size})
assert mandatory <= {Path(x["file"].replace("__","/")).name for x in saved}, "Missing mandatory Drive upload"
print("DB-free ASV results uploaded:",len(saved))
print("Drive:", "https://drive.google.com/drive/folders/"+dest)
print("SILVA taxonomic assignment: NOT RUN")
